# Fetch US Zip+4 by Addresses

This notebook demonstrates how to get the zip+4 code from US street addresses in bulk using the Smarty US Street API. It reads addresses from a CSV file and sends them in batches to the API.

## Setup

Here, we will set up the necessary imports, load environment variables for authentication, and define constants for file paths and API parameters.

In [61]:
import dotenv
import os
import pandas
import requests
import time

Create a `.env` file in the root directory of the project with the following content:

```
SMARTY_AUTH_ID=<smarty_auth_id>
SMARTY_AUTH_TOKEN=<smarty_auth_token>
```

In [62]:
dotenv.load_dotenv("../.env")

SMARTY_AUTH_ID = os.environ["SMARTY_AUTH_ID"]
SMARTY_AUTH_TOKEN = os.environ["SMARTY_AUTH_TOKEN"]

In [63]:
CSV_FILE_INPUT = "../input.csv"
CSV_FILE_OUTPUT = "../output.csv"

DATA_EMPTY_FIELD_REPLACER = "-"

PANDAS_EMPTY_FIELD_REPLACER = ""

# See https://www.smarty.com/docs/cloud/us-street-api
SMARTY_BASE_URL = "https://us-street.api.smarty.com/street-address"
SMARTY_PARAMS = { "auth-id": SMARTY_AUTH_ID, "auth-token": SMARTY_AUTH_TOKEN }
SMARTY_HEADERS = { "Content-Type": "application/json; charset=utf-8" }
SMARTY_BATCH_SIZE = 100 # maximum is 100
SMARTY_INTERVAL_IN_SECONDS = 2

## Read and Prepare Data

We will read the input CSV file containing addresses and prepare the data for processing by filling in any missing values and adding a `candidates` field to each address record. The CSV file will look like this:

```
business,street,city,state,zipcode
Smarty Example 1,165 Oakwood Ave,Cliffside Park,NJ,99705
Smarty Example 2,500 Gorge Rd Apt 2A,Cliffside Park,NJ,07010
```

In [ ]:
dataframe_input = pandas.read_csv(CSV_FILE_INPUT, dtype=str)
dataframe_input.fillna(PANDAS_EMPTY_FIELD_REPLACER, inplace=True)

dataframe_input.head()

In [65]:
data_input_list = []
for row in dataframe_input.to_dict(orient="records"):
    addons = { "candidates": 10 }
    row.update(addons)
    data_input_list.append(row)

## Request Batches to Smarty API

We will send the prepared address data to the Smarty US Street API in batches, sticking to the API's rate limits and batch size constraints.

In [ ]:
len_data_input_list = len(data_input_list)

print(f"Processing {len_data_input_list} addresses in batches of {SMARTY_BATCH_SIZE}...")

responses = []
total_batch_page = (len_data_input_list + SMARTY_BATCH_SIZE - 1) // SMARTY_BATCH_SIZE
for index in range(0, len_data_input_list, SMARTY_BATCH_SIZE):
    batch = data_input_list[index:index + SMARTY_BATCH_SIZE]
    batch_page = (index // SMARTY_BATCH_SIZE) + 1

    print(f"Processing batch {batch_page}/{total_batch_page} ({len(batch)} addresses)...")

    try:
        response = requests.post(SMARTY_BASE_URL,
                        params=SMARTY_PARAMS,
                        headers=SMARTY_HEADERS,
                        json=batch)
        response.raise_for_status()

        results = response.json()
        responses.extend(results)

        print(f"  ✓ Batch {batch_page} successful: {len(results)} addresses processed")

        if index + SMARTY_BATCH_SIZE < len_data_input_list:
            time.sleep(SMARTY_INTERVAL_IN_SECONDS)
    except Exception as error:
        print(f"  ✗ Error in batch {batch_page}: {error}")
        break

print(f"\nCompleted. Total addresses processed: {len(responses)}")

## Output Results

Finally, we will transform the API responses into a CSV by appending the zip+4 codes and other relevant information to the original addresses. The output CSV will look like this:

```
index,validation_status,street,city,state,zip,zip4
0,VALIDATED,165 Oakwood Ave,Cliffside Park,NJ,07010,07010-1126
1,VALIDATED,500 Gorge Rd Apt 2A,Cliffside Park,NJ,07010,07010-2234
```

In [ ]:
data_output_list = []

print(f"Processing {len(responses)} responses for output...")

for response in responses:
    is_validated = response.get("validation_status", "") == "VALIDATED"
    data_output_list.append({
        "index": response.get("original_index", DATA_EMPTY_FIELD_REPLACER),
        "business": response.get("original_business", DATA_EMPTY_FIELD_REPLACER),
        "validation_status": response.get("validation_status", DATA_EMPTY_FIELD_REPLACER),
        "street": response.get("delivery_line_1" if is_validated else "original_street", DATA_EMPTY_FIELD_REPLACER),
        "city": response.get("components_city_name" if is_validated else "original_city", DATA_EMPTY_FIELD_REPLACER),
        "state": response.get("components_state_abbreviation" if is_validated else "original_state", DATA_EMPTY_FIELD_REPLACER),
        "zip": response.get("components_zipcode" if is_validated else "original_zip", DATA_EMPTY_FIELD_REPLACER),
        "zip4": response.get("zip4", DATA_EMPTY_FIELD_REPLACER),
    })

dataframe_output = pandas.DataFrame(data_output_list)
dataframe_output.to_csv(CSV_FILE_OUTPUT, index=False)

print(f"  ✓ Shape: {dataframe_output.shape}")
print(f"  ✓ Columns: {dataframe_output.columns}")
if not dataframe_output.empty and "validation_status" in dataframe_output.columns:
    print(f"\nCreated. Total validated addresses: {dataframe_output['validation_status'].value_counts().get('VALIDATED', 0)}")
else:
    print("\nCreated. No validated addresses found.")